# MMIA 6013 · Taller 01

# Parte 1 — Comparación de modelos

**Objetivo:** Comparar tres modelos utilizando exactamente la misma tarea verificable y medir calidad, latencia y costo.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import time
import requests
import pandas as pd

from openai import OpenAI

from src.config import (
    OPENAI_API_KEY,
    OPENAI_ECONOMIC_MODEL,
    OPENAI_REASONING_MODEL,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    validate_environment
)

validate_environment()

client = OpenAI(api_key=OPENAI_API_KEY)

## Funciones de evaluación

In [3]:
def call_openai(model_name, prompt):

    start = time.perf_counter()

    response = client.responses.create(
        model=model_name,
        input=prompt
    )

    latency = time.perf_counter() - start

    usage = response.usage

    return {
        "prediction": response.output_text.strip(),
        "latency": latency,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens
    }

In [15]:
def call_qwen(prompt):

    start = time.perf_counter()

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:1.7b",
            "prompt": prompt,
            "stream": False
        }
    ).json()

    print(response)

    latency = time.perf_counter() - start

    return {
        "prediction": response["response"].strip(),
        "latency": latency,
        "input_tokens": response["prompt_eval_count"],
        "output_tokens": response["eval_count"]
    }

In [5]:
PRICE_TABLE = {
    "gpt-4o-mini": {
        "input": 0.15,
        "output": 0.60
    },
    "gpt-5.6": {
        "input": 0.20,
        "output": 1.20
    },
    "qwen3:1.7b": {
        "input": 0.0,
        "output": 0.0
    }
}

def compute_cost(model_name, input_tokens, output_tokens):

    prices = PRICE_TABLE[model_name]

    return (
        input_tokens / 1_000_000 * prices["input"] +
        output_tokens / 1_000_000 * prices["output"]
    )

In [6]:
PROMPT_TEMPLATE = """
You are a support ticket classifier.

Classify the ticket into exactly one category:

- HR
- Finance
- Technical

Return only the category name.

Ticket:
{ticket}
"""

In [7]:
def benchmark_model(model_name, dataframe):

    results = []

    for _, row in dataframe.iterrows():

        prompt = PROMPT_TEMPLATE.format(
            ticket=row.ticket
        )

        if model_name == "qwen3:1.7b":
            output = call_qwen(prompt)
        else:
            output = call_openai(model_name, prompt)

        prediction = output["prediction"].splitlines()[0].strip()

        cost = compute_cost(
            model_name,
            output["input_tokens"],
            output["output_tokens"]
        )

        results.append({
            "id": row.id,
            "model": model_name,
            "expected": row.expected,
            "prediction": prediction,
            "correct": prediction == row.expected,
            "latency": output["latency"],
            "input_tokens": output["input_tokens"],
            "output_tokens": output["output_tokens"],
            "cost_usd": cost
        })

    return pd.DataFrame(results)

In [8]:
cases = pd.read_csv(
    RAW_DATA_DIR / "casos.csv"
)

cases

,id,ticket,expected
0,1,Employee cannot access payroll after password ...,HR
1,2,The monthly salary was deposited twice into my...,Finance
2,3,VPN connection fails when working remotely.,Technical
3,4,Request to update bank account for direct depo...,HR
4,5,Credit card payment appears duplicated in tran...,Finance
5,6,Laptop shows a blue screen during startup.,Technical
6,7,Need an employment verification letter for vis...,HR
7,8,Interest calculation on a mortgage statement s...,Finance
8,9,Email client cannot synchronize with the corpo...,Technical
9,10,Update my home address in employee records.,HR


In [9]:
results_gpt4o = benchmark_model(
    OPENAI_ECONOMIC_MODEL,
    cases
)

results_gpt4o.head()

,id,model,expected,prediction,correct,latency,input_tokens,output_tokens,cost_usd
0,1,gpt-4o-mini,HR,Technical,False,1.802679,49,2,0.000009
1,2,gpt-4o-mini,Finance,Finance,True,0.981276,51,2,0.000009
2,3,gpt-4o-mini,Technical,Technical,True,0.920877,48,2,0.000008
3,4,gpt-4o-mini,HR,Finance,False,1.004200,50,2,0.000009
4,5,gpt-4o-mini,Finance,Finance,True,0.920384,50,2,0.000009


In [10]:
results_gpt56 = benchmark_model(
    OPENAI_REASONING_MODEL,
    cases
)

results_gpt56.head()

,id,model,expected,prediction,correct,latency,input_tokens,output_tokens,cost_usd
0,1,gpt-5.6,HR,Technical,False,1.811649,48,33,0.000049
1,2,gpt-5.6,Finance,Finance,True,1.206664,50,5,0.000016
2,3,gpt-5.6,Technical,Technical,True,1.264540,47,5,0.000015
3,4,gpt-5.6,HR,HR,True,1.333763,49,5,0.000016
4,5,gpt-5.6,Finance,Finance,True,1.116842,49,5,0.000016


In [16]:
results_qwen = benchmark_model(
    "qwen3:1.7b",
    cases
)

results_qwen.head()

{'model': 'qwen3:1.7b', 'created_at': '2026-09-20T18:59:50.5675552Z', 'response': 'Technical', 'thinking': "Okay, let's see. The user mentioned that their employee can't access payroll after resetting their password. So the main issue here is related to employee access and password reset.\n\nFirst, I need to determine which category this falls under. The options are HR, Finance, or Technical. \n\nHR usually deals with employee relations, benefits, and policies. Finance would handle payroll, accounting, and financial transactions. Technical support is for issues related to software, systems, and technical problems.\n\nThe problem here is about accessing payroll after a password reset. Payroll is a financial function, so Finance would be involved in handling payroll payments and related financial processes. The employee's ability to access payroll after a password reset is a technical issue related to system access and authentication. However, the core problem is financial because it's a

,id,model,expected,prediction,correct,latency,input_tokens,output_tokens,cost_usd
0,1,qwen3:1.7b,HR,Technical,False,18.327518,51,599,0.0
1,2,qwen3:1.7b,Finance,Finance,True,5.107280,53,164,0.0
2,3,qwen3:1.7b,Technical,Technical,True,5.333757,50,176,0.0
3,4,qwen3:1.7b,HR,Finance,False,4.763377,52,143,0.0
4,5,qwen3:1.7b,Finance,Finance,True,5.807909,52,204,0.0


In [12]:
results = pd.concat([
    results_gpt4o,
    results_gpt56,
    results_qwen
], ignore_index=True)

results.head()

,id,model,expected,prediction,correct,latency,input_tokens,output_tokens,cost_usd
0,1,gpt-4o-mini,HR,Technical,False,1.802679,49,2,0.000009
1,2,gpt-4o-mini,Finance,Finance,True,0.981276,51,2,0.000009
2,3,gpt-4o-mini,Technical,Technical,True,0.920877,48,2,0.000008
3,4,gpt-4o-mini,HR,Finance,False,1.004200,50,2,0.000009
4,5,gpt-4o-mini,Finance,Finance,True,0.920384,50,2,0.000009


In [13]:
summary = (
    results
    .groupby("model")
    .agg(
        accuracy=("correct", "mean"),
        latency=("latency", "mean"),
        total_cost=("cost_usd", "sum"),
        input_tokens=("input_tokens", "sum"),
        output_tokens=("output_tokens", "sum")
    )
    .reset_index()
)

summary["accuracy"] *= 100

summary

,model,accuracy,latency,total_cost,input_tokens,output_tokens
0,gpt-4o-mini,80.0,1.055484,0.000086,496,20
1,gpt-5.6,90.0,1.479450,0.000191,486,78
2,qwen3:1.7b,90.0,6.414043,0.000000,517,2011


In [14]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

results.to_csv(
    PROCESSED_DATA_DIR / "resultados.csv",
    index=False
)

summary.to_csv(
    PROCESSED_DATA_DIR / "parte1_summary.csv",
    index=False
)

print("Archivos exportados correctamente")

Archivos exportados correctamente


## Conclusiones

Los costos fueron calculados utilizando los precios oficiales de la tabla semestral del taller y los tokens realmente consumidos por cada modelo.

En la evaluación, GPT-5.6 y Qwen 3 1.7B obtuvieron la mayor exactitud (90 %), mientras que GPT-4o mini alcanzó 80 % con el menor costo y la menor latencia entre los modelos remotos.

Un resultado relevante fue el comportamiento de Qwen 3 1.7B: el modelo generó 2011 tokens de salida, muy por encima de los modelos de OpenAI. Esto se debe a que, mediante Ollama, Qwen devuelve su proceso de razonamiento además de la respuesta final cuando el modo thinking está habilitado. Aunque para la evaluación únicamente se utilizó la primera línea (la categoría predicha), la latencia y el conteo de output tokens incluyen todo el razonamiento generado.

En consecuencia, la comparación muestra que la calidad de la predicción fue similar entre GPT-5.6 y Qwen, pero las métricas de tiempo y longitud de salida estuvieron influenciadas por la estrategia de generación de cada modelo, especialmente por la exposición explícita del razonamiento en Qwen.